# Week 9 Pair Programming: Transfer Learning Architects

## Goal
Compare different pretrained backbones for transfer learning. Build judgment about which model to reach for in production.

## Setup
Find a partner. Driver writes code; navigator reads, asks questions, suggests changes. **Switch every 5 minutes.**

## Time
25 minutes total.

## What you'll do
Train transfer learning heads on top of 3 different backbones (MobileNetV2, ResNet50, EfficientNetB0). Compare accuracy, parameters, and inference speed.

## Continuity
Builds on `week9_live_session.ipynb`. Same flowers dataset; we just swap the backbone.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
from keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from keras.applications.resnet50 import preprocess_input as resnet_preprocess
from keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
import numpy as np
import time
import pathlib
from PIL import Image

keras.utils.set_random_seed(42)

# The flowers/ folder is built in the live session (organize_oxford_flowers.py).
# It lives next to that notebook, so point one directory over (or copy it here).
DATA_DIR = '../day1_live_session/flowers'
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

def load_split(split):
    """Read flowers/<split>/<class>/*.jpg into (images, labels) NumPy arrays — no TensorFlow."""
    root = pathlib.Path(DATA_DIR) / split
    classes = sorted(d.name for d in root.iterdir() if d.is_dir())
    images, labels = [], []
    for label_idx, cls in enumerate(classes):
        for img_path in sorted((root / cls).glob('*.jpg')):
            img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
            images.append(np.asarray(img, dtype='float32'))
            labels.append(label_idx)
    return np.array(images), np.array(labels), classes

X_train, y_train, class_names = load_split('train')
X_test, y_test, _ = load_split('test')
print(f'Classes: {class_names}')
print(f'Train: {X_train.shape[0]} images | Test: {X_test.shape[0]} images')

## Helper: build + train + benchmark any frozen-base transfer model

In [ ]:
def build_and_eval(backbone_class, preprocess_fn, name, epochs=5):
    """Build a frozen-base transfer model with the given backbone, train, evaluate, benchmark."""
    base = backbone_class(
        input_shape=(160, 160, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    inputs = keras.Input(shape=(160, 160, 3))
    x = preprocess_fn(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(class_names), activation='softmax')(x)
    model = keras.Model(inputs, outputs, name=name)

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    train_start = time.time()
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                        epochs=epochs, batch_size=BATCH_SIZE, verbose=0)
    train_time = time.time() - train_start

    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]

    # Inference benchmark — single image
    sample = X_test[:1]
    model.predict(sample, verbose=0)  # warm-up
    inf_start = time.time()
    for _ in range(20):
        model.predict(sample, verbose=0)
    inf_time_per_image = (time.time() - inf_start) / 20

    return {
        'name': name,
        'total_params': model.count_params(),
        'trainable_params': sum(p.numel() if hasattr(p, 'numel') else np.prod(p.shape)
                                for p in model.trainable_weights),
        'test_acc': test_acc,
        'train_time_sec': train_time,
        'inference_time_ms': inf_time_per_image * 1000,
    }

results = []

## Backbone 1: MobileNetV2 (lightweight, mobile-friendly)

**Driver:** run this cell. **Navigator:** while it trains, read up on MobileNetV2 architecture (depthwise separable convolutions). Note the params and accuracy.

In [ ]:
results.append(build_and_eval(MobileNetV2, mobilenet_preprocess, 'MobileNetV2'))
print(results[-1])

## Backbone 2: ResNet50 (medium-sized, classic)

**Switch roles.**

Hypothesis to discuss: ResNet50 has more parameters than MobileNetV2. Does that translate to better accuracy on this small dataset?

In [ ]:
results.append(build_and_eval(ResNet50, resnet_preprocess, 'ResNet50'))
print(results[-1])

## Backbone 3: EfficientNetB0 (modern, optimized)

**Switch roles.**

Hypothesis: EfficientNet was designed to maximize accuracy-per-parameter. Does it win on the params/accuracy tradeoff?

In [ ]:
results.append(build_and_eval(EfficientNetB0, efficientnet_preprocess, 'EfficientNetB0'))
print(results[-1])

## Comparison summary

In [ ]:
print(f'{"Model":18s} | {"Total Params":>13s} | {"Test Acc":>9s} | {"Train (s)":>10s} | {"Inf (ms)":>9s}')
print('-' * 80)
for r in results:
    print(f'{r["name"]:18s} | {r["total_params"]:>13,} | {r["test_acc"]:>9.4f} | {r["train_time_sec"]:>10.1f} | {r["inference_time_ms"]:>9.2f}')

## Discussion (5 minutes)

**Answer with your partner:**

1. **Which model gave the highest test accuracy?**
2. **Which model had the smallest parameter count?**
3. **Which model is fastest for inference?**
4. **If you were deploying this to a mobile app**, which would you pick? Why?
5. **If you were deploying to a server with no resource constraints**, which would you pick?
6. **For research / experimentation**, which would you pick?

## Solutions / typical observations

<details>
<summary>Click to reveal</summary>

- **Accuracy:** Often EfficientNet > ResNet > MobileNet, but the gap is small on this small dataset. With only 500 training images, the bottleneck is data, not model capacity.
- **Parameters:** MobileNetV2 (~2.3M) < EfficientNetB0 (~4M) < ResNet50 (~23M). 10× difference between MobileNet and ResNet.
- **Inference speed:** MobileNetV2 fastest, ResNet50 slowest, EfficientNetB0 in between (depends on hardware).
- **Mobile deployment:** MobileNetV2 — designed for it.
- **Server deployment:** Either ResNet or EfficientNet, depending on accuracy/cost tradeoff.
- **Research:** EfficientNet variants (B0-B7) give a clean knob from small to large.
</details>

## Bonus: add fine-tuning to the best backbone

Pick whichever backbone won on accuracy. Add Phase 2 fine-tuning (unfreeze top 30 layers, LR=1e-5). See if you can squeeze out another 1-2% accuracy.

**Predict first:** by how much do you think fine-tuning will improve accuracy? Then run and check.